# │ ── Podcast AI-Talk-Radio-Show

Questo notebook automatizza il workflow creativo per la produzione del podcast
Partendo da un documento di Google Docs, il sistema genera:
1. **Script Audio** ottimizzato con tecniche di PNL (Gemini).
2. **Audio Multivoce** ad alta fedelt  (Gemini TTS).
3. **Copertine Artistiche** (DALL-E 3 / Imagen).
4. **Video Trailer ASMR** per Social (OpenAI Sora/Video API) in sviluppo.
5. **Upload automatico** dei risultati su Google Drive.

---

### ⚙  Setup & Requisiti
Per utilizzare questo notebook, assicurati di aver configurato i seguenti **Secrets** in Google Colab (ᐅᐅ icona chiave):
- `GEMINI_API_KEY`: Per l'elaborazione del testo e TTS.
- `OPENAI_API_key`: Per la generazione di immagini e video.
- `FolderIDGDRIVE`: Per upload e download documenti e media.
*Nota: Questo progetto   distribuito a scopo educativo e creativo.*

In [ ]:
# @title install dipendenze

!pip install --upgrade google-genai
!pip install --upgrade google-cloud-aiplatform
!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib
!pip install --upgrade pillow==11.3.0 openai

import base64
import mimetypes
import os
import re
import struct
import wave
import sys

from openai import OpenAI
from PIL import Image
from io import BytesIO

from IPython.display import Audio
from IPython.display import Image as IPImage, display

import google.auth
from google import genai
from google.genai import types
from google.colab import userdata
from google.colab import auth

from googleapiclient.discovery import build

In [ ]:
# @title IMPOSTAZIONI | NOMI | VOCI | PARAMETRI

nome_puntata = ""

OTTIMIZZA_SCRIPT_AUDIO = True
secondi_video_trailer = "12"
drive_folder_id = userdata.get('folderIDGDrive')


voice_Maschile_name = "MARIO"  # voce italiana
voice_Femminile_name = "ELENA"  # voce italiana
#voice_Maschile = "Algieba"
#voice_Femminile = "Leda"

voice_Maschile = "Fenrir"
voice_Femminile = "Despina"

In [ ]:
# @title TESTO E NOME PUNTATE

from google.colab import auth
import google.auth
from googleapiclient.discovery import build


# Definiamo gli scope di accesso: Drive readonly (per leggere i file) e Docs readonly (per leggere i documenti)
SCOPES = ['https://www.googleapis.com/auth/drive.readonly',
          'https://www.googleapis.com/auth/documents.readonly']

# Funzione per estrarre il testo da un elemento di paragrafo
def read_paragraph_element(element):
    """Restituisce il testo contenuto in un elemento di paragrafo (ParagraphElement)."""
    text_run = element.get('textRun')
    if not text_run:
        return ''  # elementi non testuali (ad es. un break, un elemento di lista, ecc.)
    return text_run.get('content', '')

# Funzione ricorsiva per estrarre testo da una lista di Structural Elements (paragrafi, tabelle, ecc.)
def read_structural_elements(elements):
    """Estrae ricorsivamente il testo da una lista di elementi strutturali del documento Google."""
    text = ''
    for value in elements:
        # Se l'elemento è un paragrafo, estraiamo il testo da ogni elemento di paragrafo
        if 'paragraph' in value:
            for elem in value.get('paragraph').get('elements', []):
                text += read_paragraph_element(elem)
        # Se l'elemento è una tabella, iteriamo su celle e righe
        elif 'table' in value:
            for row in value.get('table').get('tableRows', []):
                cells = row.get('tableCells', [])
                for cell in cells:
                    text += read_structural_elements(cell.get('content', []))
        # Se l'elemento è un sommario (Table of Contents), estraiamo il testo dai suoi contenuti
        elif 'tableOfContents' in value:
            text += read_structural_elements(value.get('tableOfContents').get('content', []))
    return text

def list_files_in_folder_recursive(drive_service, drive_folder_id, files_list=None):
    """
    Elenca tutti i file in una cartella e nelle sue sottocartelle ricorsivamente.

    Args:
        drive_service: L'oggetto del servizio Google Drive
        folder_id: L'ID della cartella da cui iniziare la ricerca
        files_list: Lista accumulativa dei file (usata internamente per la ricorsione)

    Returns:
        Lista di tutti i file trovati
    """
    if files_list is None:
        files_list = []

    page_token = None

    while True:
        # Query per trovare tutti i file di tipo document nella cartella corrente
        results = drive_service.files().list(
            q=f"'{drive_folder_id}' in parents and trashed=false",
            orderBy="modifiedTime desc",
            pageSize=100,  # Aumentato per gestire più file
            fields="nextPageToken, files(id, name, mimeType, modifiedTime)",
            pageToken=page_token
        ).execute()

        items = results.get('files', [])
        for item in items:
            # Se è una cartella, cerca ricorsivamente al suo interno
            if item['mimeType'] == 'application/vnd.google-apps.folder':
                # Aggiungi la cartella alla lista
                #files_list.append(item)
                # Ricerca ricorsiva nella sottocartella
                list_files_in_folder_recursive(drive_service, item['id'], files_list)
            else:
                # Aggiungi il file alla lista
                if item['mimeType'] == 'application/vnd.google-apps.document':
                 files_list.append(item)

        # Gestione della paginazione
        page_token = results.get('nextPageToken')
        if not page_token:
            break

    return files_list

# Autenticazione tramite browser: comparirà un link da cliccare per autorizzare l'accesso
auth.authenticate_user()

# Otteniamo le credenziali OAuth2 autenticate con gli scope specificati
creds, _ = google.auth.default(scopes=SCOPES)

# Creiamo il servizio Drive API con le credenziali ottenute
drive_service = build('drive', 'v3', credentials=creds)

# TODO: Inserisci l'ID della cartella da cui vuoi elencare i file
# Puoi trovare l'ID della cartella nell'URL di Google Drive quando apri la cartella
  # Sostituisci con l'ID della cartella desiderata
all_files = list_files_in_folder_recursive(drive_service, drive_folder_id)

if all_files:
    print(f"File Google Documenti trovati ({len(all_files)}):")
    print(f" ")
    for idx, file in enumerate(all_files, start=1):
        parents = file.get('parents', [])
        #Mostra gli ID dei genitori (cartelle) per ogni file. Questo aiuta a capire la gerarchia.
        #parent_info = f" (Genitori ID: {', '.join(parents)})" if parents else " (Cartella Root)"
        print(f"{idx}. {file['name']}")

    print(f" ")
    # Input da parte dell'utente per scegliere uno dei file elencati
    while True: # Loop per richiedere input finché non è valido
        try:
            # Corrected the length check to use all_files instead of files
            pt_corrente = int(input(f"Inserisci il numero della puntata corrente (1-{len(all_files)}): "))
            pt_successiva = int(input(f"Inserisci il numero puntata successiva (1-{len(all_files)}): "))
            # Check if the chosen numbers are within the valid range
            # Corrected the length check to use all_files instead of files
            if 1 <= pt_corrente <= len(all_files) and 1 <= pt_successiva <= len(all_files):
                break # Valid input, exit the loop
            else:
                print(f"Errore: Inserisci numeri tra 1 e {len(all_files)}.")
        except ValueError:
            print("Errore: Input non valido. Inserisci un numero intero.")

    # Recuperiamo ID e nome del file selezionato per la puntata corrente
    # Corrected to use all_files instead of files
    pt_corrente_selected = all_files[pt_corrente - 1]
    pt_corrente_id = pt_corrente_selected['id']
    pt_corrente_name = pt_corrente_selected['name']
    print(f" ")
    print(f"Puntata corrente selezionata: {pt_corrente_name} (ID: {pt_corrente_id})")

     # Recuperiamo ID e nome del file selezionato per la puntata successiva
    # Corrected to use all_files instead of files
    pt_successiva_selected = all_files[pt_successiva - 1]
    pt_successiva_id = pt_successiva_selected['id']
    pt_successiva_name = pt_successiva_selected['name']
    print(f" ")
    print(f"Puntata successiva selezionata: {pt_successiva_name} (ID: {pt_successiva_id})")

    # Creiamo il servizio Docs API con le credenziali ottenute
    docs_service = build('docs', 'v1', credentials=creds)

    # Google Docs API per ottenere la struttura del documento corrente
    pt_corrente_document = docs_service.documents().get(documentId=pt_corrente_id).execute()

    # Estrae il titolo del pt corrente
    pt_curr_nome_puntata = pt_corrente_document.get('title')
    print(f"Titolo del puntata corrente: {pt_curr_nome_puntata}")

    # Estrae il contenuto testuale dall'oggetto document corrente
    pt_curr_content = pt_corrente_document.get('body').get('content', [])
    pt_curr_testo_orig = read_structural_elements(pt_curr_content)

    # Google Docs API per ottenere la struttura del document successivo
    pt_suc_document = docs_service.documents().get(documentId=pt_successiva_id).execute()

    # Estrae il titolo del pt successiva
    pt_suc_nome_puntata = pt_suc_document.get('title')
    print(f"Titolo del puntata successivo: {pt_suc_nome_puntata}")

    # Estrae il contenuto testuale dall'oggetto document successivo
    pt_suc_content = pt_suc_document.get('body').get('content', [])
    pt_suc_testo_orig = read_structural_elements(pt_suc_content)
else:
    print("Nessun file trovato.")

In [ ]:
# @title PAROLE CHIAVE

# *** Configurazione API ***
# Imposta la tua API key Gemini (puoi caricarla da variabile d'ambiente o inserirla direttamente)
from google.colab import userdata
from google import genai

Google_API_KEY = userdata.get('GEMINI_API_KEY')
if Google_API_KEY is None or Google_API_KEY.startswith("INSERISCI"):
   print(f"Google_API_KEY non impostata")
   raise
else:
 # Inizializza il client Gemini
 client = genai.Client(api_key=Google_API_KEY)

def estrazione_parole_chiave(testo, nome_puntata):
 prompt_parole_chiave = "Analizza il seguente testo. Estrai gli argomenti trattati. Rispondi solo con un elenco separato da virgole con le keyword, concetti e termini chiave presenti nel testo proposto, ed un riassunto di al massimo 4 frasi da usare come copy nel materiale pubblicitario. Non usare nelle keyword, concetti e termini chiave con significato negativo. Ecco il testo: "+testo

 # Chiamata al modello di Geminai
 risposta_parole_chiave_testo = client.models.generate_content(
  model="gemini-3-pro-preview",
  contents=prompt_parole_chiave
 )

 # Estrae il testo ottimizzato dalla risposta
 parole_chiave_testo = risposta_parole_chiave_testo.result if hasattr(risposta_parole_chiave_testo, "result") else risposta_parole_chiave_testo.text
 print()

 if parole_chiave_testo:
  nome_file_txt = "parole_chiave_"+nome_puntata+".txt"
  print(f"parole chiave estratte: '{parole_chiave_testo}'")
  try:
    with open(nome_file_txt, "w", encoding="utf-8") as f:
     f.write(parole_chiave_testo)
    print(f"parole chiave salvate correttamente nel file '{nome_file_txt}'")
    return parole_chiave_testo
  except Exception as e:
    print(f"Errore durante il salvataggio del file: {e}")
    raise
 else:
  print("Generazione parole chiave podcast fallita.")
  raise

if 'pt_curr_testo_orig' not in globals():
 print("Nessun pt_curr_testo_orig per estrarre parole chiave")
 raise
else:
 pt_curr_parole_chiave = estrazione_parole_chiave(pt_curr_testo_orig, pt_curr_nome_puntata)


In [ ]:
# @title PROMPT GENERATIVE AI
# GESTIONE Prompt

if 'pt_curr_nome_puntata' not in globals():
    pt_curr_nome_puntata = "episodio_podcast_parole_in_viaggio"

if 'pt_curr_testo_orig' not in globals():
    print("Nessun testo da pt_curr_testo_orig.")
    raise

if 'pt_curr_parole_chiave' not in globals():
    print("Nessun testo da pt_curr_parole_chiave.")
    raise

prompt_genera_titolo ="""Ruolo
Agisci come Head of Editorial & SEO per un podcast di self-improvement in italiano (tone of voice: poetico, intimo, lucido, zero clickbait).

Obiettivo
Genera titoli episodio che combinino una frase evocativa (prima) e un benefit/keyword ad alta intenzione (dopo), separati da una barra verticale |. Massimizza CTR su Spotify Search/Autoplay mantenendo coerenza di brand.

Input
	•	Tema/insight dell’episodio: {tema}
	•	Citazione o frase guida (se presente): {citazione}
	•	Emozione target (1–2): {emozione}
	•	Keyword cluster (3–6 parole, intent-based): {keyword_cluster}
	•	Aneddoto/luogo (facoltativo): {aneddoto}
	•	Vincoli lunghezza: titolo ≤ 64 caratteri ideale (accetta fino a 70), fornisci anche una variante SHORT ≤ 42
	•	Opzione numero puntata: {numero_puntata} (se valorizzato, appendi in coda: – Puntata {numero_puntata})

Stile & Regole
	1.	Lingua: italiano.
	2.	Struttura: Parte poetica | keyword/benefit.
	3.	Evita: virgolette, emoji, hashtag, full caps, punti finali.
	4.	Capitalizzazione naturale (solo iniziale maiuscola, nomi propri).
	5.	No clickbait; prometti ciò che mantieni.
	6.	Coerenza brand: cura della parola, profondità, gentilezza; niente tono guru.
	7.	Keyword: 1–3 parole chiare a destra del | (es. ansia, resilienza, prospettiva, identità, radici).
	8.	Non ripetere la stessa keyword in più di una proposta.
	9.	Se c’è una citazione famosa, distillala: non riportarla per intero nel titolo.
	10.	Se {numero_puntata} è vuoto, non inserire numeri.

Criteri di qualità (self-check interno)
	•	Chiarezza (capibile in 2s)
	•	Hook emotivo (curiosità senza sensazionalismo)
	•	Intent match con il cluster keyword
	•	Brevità (target 48–64 caratteri)
	•	Unicità (niente eco di titoli recenti)

Output richiesto
	•	Fornisci 3 proposte + per ognuna una variante SHORT.
	•	Per ogni proposta: mostra conteggio caratteri tra parentesi e una nota di 4–7 parole sul perché funziona.
	•	Ordinale per “Hook score” (1–5) calcolato da te.
	•	In fondo, suggerisci 2 keyword alternative non usate ma coerenti.

Esempi dello stile desiderato
	•	Il dono del non ottenere | Trasformare delusioni in fortuna
	•	Ricordati chi sei | Crescere senza perdere il centro
	•	Quel no che apre strade | Prospettiva e resilienza
	•	Oltre senza smarrirsi | Identità e radici

Genera ora usando questi dati: """

prompt_genera_testo_podcast = """Sei un senior copywriter specializzato in podcast storytelling e PNL.

Obiettivo: trasformare il testo in uno script per un podcast a due voci, dal ritmo avvincente, presentando le informazioni in modo coinvolgente e dinamico.

Istruzioni operative:
      1.      Speaker
        •       Alterna rigorosamente le battute tra """+voice_Maschile_name+""": (voce maschile) e """+voice_Femminile_name+""": (voce femminile).
        •       Ogni battuta: massimo 3 – 4 frasi, per mantenere un ritmo alto ed incalzante ed il grado di attenzione dell'ascoltatore sempre alto.
      2.      Apertura fissa
        •       Quabndo trovi la parola INTRO sosituiscila con: «Ciao io sono """+voice_Maschile_name+""", ed io sono """+voice_Femminile_name+""". benvenuto a “Parole in Viaggio”, il podcast che prende ispirazione dall’omonimo libro, che trasforma semplici frasi raccolte in 59 paesi in giro per il mondo in spunti di riflessione
                Quabndo trovi la parola MARKETTA sosituiscila un breve invito non invadente a condividere, lasciare un commento, mettere 5 stelle alla puntata perchè anche altre persone possano trarne ispirazione.
      3.      Chiusura fissa
        •       Crea un trailer della puntata succesiva per creare interesse e curiosità delle puntata successiva, usando storytelling con ganci a 3 livelli (curiosità, valore, call-to-action) massimo 55 parole, i cui temi trattati sono: """+pt_curr_parole_chiave+""".
        •       Termina lo script con la battuta: «Grazie per averci ascoltato, questo è Parole in Viaggio il podcast che trasforma frasi erranti in nuove direzioni interiori. Non dimenticare di mettere 5 stelle alla puntata, condividere la tua frase preferita e lasciare un commento con #ParoleInViaggio ! Perchè anche altre persone possano trarne ispirazione! Ci vediamo sull’altro orizzonte!!!!
      4.      PNL & engagement
        •       Integra ancore verbali, linguaggio sensoriale (V-A-K), pattern di Milton e domande evocative.
        •       Usa parole-chiave ricorrenti per rinforzare l’ancoraggio emotivo.
      5.      Stile
        •       Mantieni intatto contenuto e significato del testo da adattare.
        •       Registro colloquiale, acronimi e anglicismi quando pertinenti.
        •       Usa micro incongruenze linguistiche per attirare l'attenzione quando pertinenti.
      6.      Output
        •      Inizia ogni battuta con """+voice_Maschile_name+""" o """+voice_Femminile_name+""", senza * o altra marcatura.
        •      Fornisci solo lo script finito.
⸻

Testo da adattare: """+ pt_curr_testo_orig

print(f"prompt per ottimizzazione testo")
print(prompt_genera_testo_podcast)

prompt_generazione_audio = "Leggi il testo in italiano, leggermente accellerato, con accento italiano e un ritmo dinamico per tenere alta l'attenzione e l'interesse. Ottimizza le due voci in modo da rendele uguali a speaker radiofonico, utilizzando in modo opportuno lo stile, il tono, l'accento e il ritmo delle voci. Rendi infine l'audio perfetto per un podcast audio a 2 voci, non distiguibile da un podcast radiofonifico. \n"

print(f"prompt per generazione audio ")
print()

prompt_immagine_copertina = (
    "Crea un immagine per una copertina virale di un podcast dal nome parole in viaggio. " +
    "Ecco le keyword da cui prendere ispirazione: "+pt_curr_parole_chiave+". "+
    "Stile mix tra cubismo e bauhaus. Soggetto principale simbolico, con forti contrasti. "+
    "La palette di colori rispecchia le parole chiave, sfondo semplice. "+
    "Composizione dinamica, basata sulle keyword che hai a disposizione, visivamente accattivante, che attiri l'attenzione e crei curiosità  nell'ascoltatore e stimoli all' ascolto."+
    "Usa storytelling con ganci a 3 livelli (curiosità, valore, call-to-action). "+
    "Copertina leggibile anche in formato thumbnail, . "+
    "Nella parte alta inserisci il titolo della puntata rendendolo virale e leggermente clickbate con il testo tutto in maiuscolo: "+pt_corrente_name+ ". "+
    "Nella parte bassa inserisci il nome del pocast in maiuscolo. "+
    "Rispetta le linee guida e best practice di Spotify, le specifiche tecniche: 3000×3000 px, 72 DPI, RGB, formato JPG. "

)

print(f"prompt per generazione immagine ")
print(" ")

promt_video_trailer = ("Crea il video trailer della durata di "+secondi_video_trailer+" secondi per la puntata del podcast intitolata "+pt_curr_nome_puntata+". Stile ASMR accattivante."+
    "(chiedi consigli agli esperti di Meta per strutturare lo script del video con gli hook giusti ed efficaci utilizzando frasi ad effetto (hook) per catturare l'attenzione su Instagram.). "+
    "Concentrati sui concetti chiave: "+pt_curr_parole_chiave+". "+
    "Pubblico di riferimento a cui si rivolge: Utenti di Instagram interessati a crescita personale, filosofia e viaggi."
    "Soggetto principale simbolico, con forti contrasti. Palette di colori che rispecchino le parole chiave fornite, su sfondo semplice. "+
    "Composizione d’impatto, basata sulle keyword e concetti forniti, visivamente accattivante, che attiri l'attenzione, clickbait, dell'ascoltatore. "+
    "Il parlato deve essere letto entro la durata del video ("+secondi_video_trailer+" secondi), nessun audio tagliato o troncato, deve sempre finire con l'invito a cliccare per ascoltare la puntata intera. "+
    "Aggiungi musica di sottofondo energica e di tendenza, adatta per un reel virale. "+
    "Durante tutto il video, nella parte bassa inserisci il titolo della puntata con il testo tutto in maiuscolo: "+pt_corrente_name+ ". "
)
print(f"prompt per generazione video trailer")
print(" ")

prompt_negative_immagine = ("watermark, logo, cgi, 3D, artwork")

In [ ]:
# @title SCRIPT per puntata PODCAST

if 'prompt_genera_testo_podcast' not in globals():
    print("Nessun testo per elaborare la puntata")
    raise

# *** Configurazione API ***
# Imposta la tua API key Gemini (puoi caricarla da variabile d'ambiente o inserirla direttamente)
Google_API_KEY = userdata.get('GEMINI_API_KEY')
if Google_API_KEY is None or Google_API_KEY.startswith("INSERISCI"):
    print(f"Google_API_KEY non impostata")
    raise

# Inizializza il client Gemini
client = genai.Client(api_key=Google_API_KEY)

if OTTIMIZZA_SCRIPT_AUDIO:

    # Chiamata al modello text-gen di Gemini (usiamo il modello Pro con contesto esteso, compatibile con ~32k token)
    risposta_testo = client.models.generate_content(
        model="gemini-3-pro-preview",  # modello Gemini 2.5 Pro (testo) - versione preview corrente
        contents=prompt_genera_testo_podcast
    )
    # Estrae il testo ottimizzato dalla risposta
    testo_podcast_audio = risposta_testo.result  if hasattr(risposta_testo, "result") else risposta_testo.text
    print()
    if testo_podcast_audio:
        nome_file_txt = "script_podcast_"+pt_curr_nome_puntata+".txt"
        try:
          with open(nome_file_txt, "w", encoding="utf-8") as f:
            f.write(testo_podcast_audio)
          print(f"Testo salvato correttamente nel file '{nome_file_txt}'")
        except Exception as e:
          print(f"Errore durante il salvataggio del file: {e}")
          raise
    else:
        print("Generazione testo podcast fallita.")
        raise



In [ ]:
# @title GENERAZIONE AUDIO PUNTATA

from google.genai import types
import mimetypes
import struct # Import the struct module

if 'testo_podcast_audio' not in globals():
  print("Nessun testo da leggere.")
  raise

# Removed the concatenation with instructional text. Only the dialogue is passed.
# prompt_generazione_audio += testo_podcast_audio
print(f"prompt per generazione audio")
# The variable 'prompt_generazione_audio' now only holds the instructions.
# Printing it here would be misleading as it's not the content used for audio gen.
# For clarity, let's print the actual content being sent for audio generation.
print(f"Content being sent for audio generation:\n{testo_podcast_audio}")
print()

def convert_to_wav(audio_data: bytes, mime_type: str) -> bytes:
    """Generates a WAV file header for the given audio data and parameters.

    Args:
        audio_data: The raw audio data as a bytes object.
        mime_type: Mime type of the audio data.

    Returns:
        A bytes object representing the WAV file header.
    """
    parameters = parse_audio_mime_type(mime_type)
    bits_per_sample = parameters["bits_per_sample"]
    sample_rate = parameters["rate"]
    num_channels = 1
    data_size = len(audio_data)
    bytes_per_sample = bits_per_sample // 8
    block_align = num_channels * bytes_per_sample
    byte_rate = sample_rate * block_align
    chunk_size = 36 + data_size  # 36 bytes for header fields before data chunk size

    # http://soundfile.sapp.org/doc/WaveFormat/

    header = struct.pack(
        "<4sI4s4sIHHIIHH4sI",
        b"RIFF",          # ChunkID
        chunk_size,       # ChunkSize (total file size - 8 bytes)
        b"WAVE",          # Format
        b"fmt ",          # Subchunk1ID
        16,               # Subchunk1Size (16 for PCM)
        1,                # AudioFormat (1 for PCM)
        num_channels,     # NumChannels
        sample_rate,      # SampleRate
        byte_rate,        # ByteRate
        block_align,      # ByteAlign
        bits_per_sample,  # BitsPerSample
        b"data",          # Subchunk2ID
        data_size         # Subchunk2Size (size of audio data)
    )
    return header + audio_data

def parse_audio_mime_type(mime_type: str) -> dict[str, int | None]:
    """Parses bits per sample and rate from an audio MIME type string.

    Assumes bits per sample is encoded like "L16" and rate as "rate=xxxxx".

    Args:
        mime_type: The audio MIME type string (e.g., "audio/L16;rate=24000").

    Returns:
        A dictionary with "bits_per_sample" and "rate" keys. Values will be
        integers if found, otherwise None.
    """
    bits_per_sample = 16
    rate = 24000

    # Extract rate from parameters
    parts = mime_type.split(";")
    for param in parts: # Skip the main type part
        param = param.strip()
        if param.lower().startswith("rate="):
            try:
                rate_str = param.split("=", 1)[1]
                rate = int(rate_str)
            except (ValueError, IndexError):
                # Handle cases like "rate=" with no value or non-integer value
                pass # Keep rate as default
        elif param.startswith("audio/L"):
            try:
                bits_per_sample = int(param.split("L", 1)[1])
            except (ValueError, IndexError):
                pass # Keep bits_per_sample as default if conversion fails

    return {"bits_per_sample": bits_per_sample, "rate": rate}

def save_binary_file(file_name, data):
    f = open(file_name, "wb")
    f.write(data)
    f.close()
    print(f"File saved to to: {file_name}")

Google_API_KEY = userdata.get('GEMINI_API_KEY')
if Google_API_KEY is None or Google_API_KEY.startswith("INSERISCI"):
    print(f"Google_API_KEY non impostata")
    raise

# Configurazione delle voci per ciascuno speaker (utilizziamo voci italiane esistenti)
# I nomi 'Speaker A' e 'Speaker B' devono corrispondere al testo_dialogo.
multi_voice_config = types.MultiSpeakerVoiceConfig(
    speaker_voice_configs = [
        types.SpeakerVoiceConfig(
            speaker=voice_Maschile_name,  # deve combaciare con le etichette nel testo
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name=voice_Maschile)
            )
        ),
        types.SpeakerVoiceConfig(
            speaker=voice_Femminile_name,
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name=voice_Femminile)
            )
        )
    ]
)

# Impostazione della configurazione di generazione:
gen_config = types.GenerateContentConfig(
    response_modalities=["AUDIO"],       # Vogliamo output audio
    speech_config=types.SpeechConfig(
        multi_speaker_voice_config = multi_voice_config
    )
)

# Generazione dell'audio tramite Gemini 2.5 Pro TTS
client = genai.Client(api_key=Google_API_KEY)
response = client.models.generate_content(
    model="gemini-2.5-pro-preview-tts",  # Modello TTS multilingua ad alte prestazioni (voce di qualità)
    contents=testo_podcast_audio, # Pass only the dialogue content
    config=gen_config
)

# Estrazione dei dati audio dalla risposta.
# La risposta contiene candidati; prendiamo il primo candidato e il primo contenuto audio inline.
audio_data = None

try:
    #audio_data = response.candidates[0].content.parts[0].inline_data.data

    # Salvataggio su file WAV
    output_wav = pt_curr_nome_puntata
    #salva_audio_wav(output_wav, audio_data)

    # Assuming 'response' is the variable containing the response from the API call
    # Iterate through candidates and parts to find inline_data
    inline_data = None
    for candidate in response.candidates:
        if hasattr(candidate, 'content') and hasattr(candidate.content, 'parts'):
            for part in candidate.content.parts:
                if hasattr(part, 'inline_data'):
                    inline_data = part.inline_data
                    break  # Found the inline_data, no need to search further
        if inline_data:
            break # Found the inline_data, no need to search further

    if inline_data:
        print(f"AUDIO GENERATO")

        data_buffer = inline_data.data
        file_extension = mimetypes.guess_extension(inline_data.mime_type)
        if file_extension is None:
            file_extension = ".wav"
            data_buffer = convert_to_wav(inline_data.data, inline_data.mime_type)
        save_binary_file(f"{pt_curr_nome_puntata}{file_extension}", data_buffer)

        if output_wav and os.path.exists(output_wav):
            display(Audio(output_wav))
        else:
            print(f"File audio '{output_wav}' not found.")
    else:
        print("No inline audio data found in the response.")


except Exception as e:
    # In caso di formato leggermente diverso (dipende dalla versione SDK), tentiamo altri attributi:
    audio_data = getattr(response, "audio", None) or getattr(response, "content", None)
    if audio_data is None:
      print(f"Generazione audio fallita")
      raise

In [ ]:
# @title COPERTINA gpt-image-1 OPEN AI

print(f"prompt per generazione immagine")
print(f"{prompt_immagine_copertina}")

if 'negative_prompt' not in globals():
    # It seems negative_prompt was not defined earlier, so we define it here.
    negative_prompt = ""
    print(f"negative_prompt non definito, impostato a '{negative_prompt}'")

# --- CONFIG --------------------------------------------------------------
OPENAI_API_KEY = userdata.get('OPENAI_API_key')
if OPENAI_API_KEY is None or OPENAI_API_KEY.startswith("INSERISCI"):
    print(f"OPENAI_API_KEY non impostata")
    raise

from openai import OpenAI
from PIL import Image
from io import BytesIO
import base64

client = OpenAI(api_key=OPENAI_API_KEY)

#Generate image via OpenAI Images API and download it locally."""
result = client.images.generate(
    model="gpt-image-1",
    prompt=prompt_immagine_copertina,
    size="1024x1024",
    quality="high",
    n=3
)

i = 1

for data in result.data:
  img_path="copertina_"+pt_curr_nome_puntata+"_"+str(i)+".jpg"
  print(f"Immagine generata '{img_path}")

  image_base64 = data.b64_json
  image_bytes = base64.b64decode(image_base64)

  # Adjust this if you want a high-quality
  image = Image.open(BytesIO(image_bytes))

  # Check if the image is in RGBA mode and convert to RGB if necessary.
  # JPEG format does not support the alpha channel.
  #print
  if image.mode == 'RGBA':
    # Create a new background image with a white background
    # You can change 'white' to any color or 'black' if preferred
    background = Image.new('RGB', image.size, (255, 255, 255))
    # Paste the RGBA image onto the background. The alpha channel
    # will be used to blend the image with the background.
    background.paste(image, mask=image.split()[3]) # Use the alpha channel as mask
    image = background # Now the image is in RGB mode
  image.save(img_path, format="JPEG", quality=100, optimize=True)
  print(f"Immagine salvata correttamente nel file '{img_path}'")
  i = i+1

In [ ]:
import base64
import mimetypes
import os
from google import genai
from google.genai import types
from google.colab import userdata


def save_binary_file(file_name, data):
    f = open(file_name, "wb")
    f.write(data)
    f.close()
    print(f"File saved to to: {file_name}")


def generate():
    client = genai.Client(
        api_key=Google_API_KEY,
    )

    model = "gemini-2.5-flash-image"
    contents = [
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(text=prompt_immagine_copertina),
            ],
        ),
    ]
    generate_content_config = types.GenerateContentConfig(
        temperature=0.75,
        response_modalities=[
            "IMAGE"
        ],
    )

    file_index = 0
    for chunk in client.models.generate_content_stream(
        model=model,
        contents=contents,
        config=generate_content_config,
    ):
        if (
            chunk.candidates is None
            or chunk.candidates[0].content is None
            or chunk.candidates[0].content.parts is None
        ):
            continue
        if chunk.candidates[0].content.parts[0].inline_data and chunk.candidates[0].content.parts[0].inline_data.data:
            file_name = f"ENTER_FILE_NAME_{file_index}"
            file_index += 1
            inline_data = chunk.candidates[0].content.parts[0].inline_data
            data_buffer = inline_data.data
            file_extension = mimetypes.guess_extension(inline_data.mime_type)
            save_binary_file(f"{file_name}{file_extension}", data_buffer)
        else:
            print(chunk.text)

print(f"prompt per generazione immagine")
print(f"{prompt_immagine_copertina}")

# --- CONFIG --------------------------------------------------------------
# The Google_API_KEY is already defined in a previous cell.
# No need to fetch it again or check it here.
generate()


In [ ]:
from IPython.display import Image as IPImage, display

# Inizializza il client Gemini
client = genai.Client(api_key=Google_API_KEY)

print(f"prompt per generazione immagine")
print(f"{prompt_immagine_copertina}")

if 'negative_prompt' not in globals():
    # It seems negative_prompt was not defined earlier, so we define it here.
    negative_prompt = ""
    print(f"negative_prompt non definito, impostato a '{negative_prompt}'")

# --- CONFIG --------------------------------------------------------------
# The Google_API_KEY is already defined in a previous cell.
# No need to fetch it again or check it here.

MODEL_ID = "imagen-4.0-ultra-generate-001"

number_of_images = 3
person_generation = "ALLOW_ADULT"
aspect_ratio = "1:1"

#Generate image via google GENIMAGE 4 API and download it locally."
result = client.models.generate_images(
    model=MODEL_ID,
    prompt=prompt_immagine_copertina,
    config=dict(
        number_of_images=number_of_images,
        output_mime_type="image/jpeg",
        person_generation=person_generation,
        aspect_ratio=aspect_ratio
    )
)

i = 1
for generated_image in result.generated_images:
  # Display the image using IPython.display.Image
  # Access the image data bytes directly
  image_bytes = generated_image.image.image_bytes

  # Display the image using IPython.display.Image
  display(IPImage(data=image_bytes))


  img_path="copertina_"+pt_curr_nome_puntata+"_"+str(i)+".jpg"
  print(f"Immagine generata '{img_path}")

  # Adjust this if you want a high-quality
  image = Image.open(BytesIO(image_bytes))

  # Check if the image is in RGBA mode and convert to RGB if necessary.
  # JPEG format does not support the alpha channel.
  print
  if image.mode == 'RGBA':
    # Create a new background image with a white background
    # You can change 'white' to any color or 'black' if preferred
    background = Image.new('RGB', image.size, (255, 255, 255))
    # Paste the RGBA image onto the background. The alpha channel
    # will be used to blend the image with the background.
    background.paste(image, mask=image.split()[3]) # Use the alpha channel as mask
    image = background # Now the image is in RGB mode
  image.save(img_path, format="JPEG", quality=100, optimize=True)
  print(f"Immagine salvata correttamente nel file '{img_path}'")
  i = i+1

In [ ]:

# @title Creazione script per Trailer Video

# Inizializza il client Gemini

promt_video_trailer_ok = str(promt_video_trailer)

if 'testo_podcast_audio' in globals():
 promt_video_trailer_ok = promt_video_trailer_ok + "Ecco il testo dello script da cui prendere ispirazione: "+testo_podcast_audio

promt_video_trailer_ok="TRaduci in inglese corrente il seguenre testo"
if promt_video_trailer_ok:
        nome_file_txt = "prompt_trailer_"+pt_curr_nome_puntata+".txt"
        try:
          with open(nome_file_txt, "w", encoding="utf-8") as f:
            f.write(promt_video_trailer_ok)
          print(f"prompt trailer correttamente nel file '{nome_file_txt}'")
        except Exception as e:
          print(f"Errore durante il salvataggio del prompt trailer: {e}")
          raise
else:
        print("Generazione prompt trailer fallita.")
        raise


client = genai.Client(api_key=Google_API_KEY)


# Chiamata al modello text-gen di Gemini (usiamo il modello Pro con contesto esteso, compatibile con ~32k token)
risposta_testo = client.models.generate_content(
        model="gemini-2.5-pro",  # modello Gemini 2.5 Pro (testo) - versione preview corrente
        contents=promt_video_trailer_ok
    )
    # Estrae il testo ottimizzato dalla risposta
    testo_podcast_audio = risposta_testo.result  if hasattr(risposta_testo, "result") else risposta_testo.text
    print()
    if testo_podcast_audio:
        nome_file_txt = "script_podcast_"+pt_curr_nome_puntata+".txt"
        try:
          with open(nome_file_txt, "w", encoding="utf-8") as f:
            f.write(testo_podcast_audio)
          print(f"Testo salvato correttamente nel file '{nome_file_txt}'")
        except Exception as e:
          print(f"Errore durante il salvataggio del file: {e}")
          raise
    else:
        print("Generazione testo podcast fallita.")
        raise

In [ ]:
import os
import wave
from google import genai
from google.genai import types
from google.colab import userdata
from openai import OpenAI
import time

if 'promt_video_trailer' not in globals():
    print("Nessun testo per elaborare lo script il trailer")

promt_video_trailer_ok="""CONCEPT E MOOD GENERALE Titolo: PT 18 – L’orizzonte delle infinite possibilità
Durata: {{secondi_video_trailer}} secondi
Stile: ASMR narrativo e poetico, ritmo morbido e magnetico
Formato: Verticale 9:16 (TikTok / Reels / Shorts)
Tono: Caldo, contemplativo, autentico, ispirazionale

Descrizione concettuale:
Il trailer mostra due podcaster (ELENA e MARIO), seduti in uno studio intimo con luci calde, microfoni e cuffie. La camera alterna primi piani dei loro volti e mani che si muovono mentre parlano, con immagini immersive della natura — il mare, le stelle, l’alba, il vento, la sabbia, la luce che entra da una finestra.
Il contrasto tra interno e esterno, voce e silenzio, rappresenta l’incontro tra parola e immaginazione, realtà e possibilità.

Keyword guida: orizzonte, possibilità, scelta, creazione, cambiamento, esplorazione, connessione, coraggio, esistenza, rinascita.

Palette visiva: blu profondo, oro tenue, sabbia chiara, bianco latteo, riflessi caldi.
Soggetto simbolico: il contrasto tra voce interiore e spazio infinito — rappresentato visivamente dal respiro dei due speaker e dalle immagini cosmiche o naturali che scorrono.

Titolo fisso in basso per tutta la durata del video:
PT 18 – L’ORIZZONTE DELLE INFINITE POSSIBILÀ
(font elegante, bianco, leggero effetto glow, grandezza del font 17px).

LINEE GUIDA PER STRUTTURA E VIRALITÀ (secondo Meta)

Hook nei primi 2 secondi: domanda profonda, detta in ASMR, con close-up ravvicinato.
Ritmo sensoriale: alternanza visiva costante (speaker → natura → speaker → luce).
Voce in 3D stereo: ELENA a sinistra, MARIO a destra.
Sussurro, respiro, micro movimenti labiali ben visibili.
Suoni ambientali: vento, respiro, nota sintetica, eco marino.
Obiettivo virale: trasmettere connessione emotiva immediata e curiosità spirituale.
Durata precisa: {{secondi_video_trailer}} secondi, chiusura dolce con invito ad ascoltare l’intera puntata.

SCRIPT PARLATO + MONTAGGIO VISIVO ({{secondi_video_trailer}} secondi totale)

[0:00–0:02] – ELENA
In studio, luce calda, microfono visibile, close-up sul viso. Lieve eco ASMR.
“E se la vita non fosse una linea retta… ma un orizzonte completamente aperto?”
Taglio su onde lente al tramonto, dissolvenza sul riflesso del sole.

[0:03–0:05] – MARIO
In studio, cuffie, sfondo scuro, controluce su profilo.
“Uno spazio senza confini… dove ogni possibilità che immagini può davvero esistere.”
Taglio su cielo stellato, galassia che si espande.

[0:05–0:07] – ELENA e MARIO (insieme, alternati in split screen)
Si guardano, leggeri sorrisi, luce calda e naturale.
“Benvenuto a Parole in Viaggio, il podcast che trasforma semplici frasi raccolte in 59 paesi in spunti di riflessione.”
Transizione con immagini di vento tra l’erba, mani che sfiorano la luce, respiro profondo.

[0:07–0:10] – MARIO (in chiusura)
Voce profonda, viso ravvicinato, luce dorata sul microfono.
“Clicca e ascolta l’episodio completo. Scopri il tuo orizzonte.”
Dissolvenza finale sul mare o su una linea d’orizzonte che si allarga.

Sound design consigliato:
Suono di vento + basso ambientale etereo.
Battito lento o eco di respiro.
Nessuna musica invadente, ma tappeto sonoro continuo che accompagna la voce."""

# --- CONFIG --------------------------------------------------------------
OPENAI_API_KEY = userdata.get('OPENAI_API_key')
if OPENAI_API_KEY is None or OPENAI_API_KEY.startswith("INSERISCI"):
    print(f"OPENAI_API_KEY non impostata")
    raise

client = OpenAI(api_key=OPENAI_API_KEY)

# The `with open("reference_frame.png", "rb") as img:` block is not used for video generation.
# It has been commented out to prevent errors and ensure correct execution.
# with open("reference_frame.png", "rb") as img:

video = client.videos.createAndPoll(
    prompt=str(promt_video_trailer_ok), # Convert the set to a string
    seconds=int(secondi_video_trailer),
    size="720x1280"
    # Add input_reference if you have a video ID to use as input
    # input_reference=input_reference
)

print(f"Video creation initiated with ID: {video.id}")

# Check video status periodically
video_id = video.id
status = None
while status != "completed":
    try:
        video_status = client.videos.retrieve(video_id=video_id)
        status = video_status.status
        print(f"Video status: {status} | {video_status.progress}%")
        if status == "failed":
            print("Video creation failed.")
            break
        if status != "completed":
            time.sleep(15) # Wait for 5 seconds before checking again
    except Exception as e:
        print(f"Error checking video status: {e}")
        break

if status == "completed":
    try:
        responseVid = client.videos.download_content(
            video_id=video_id,
        )
        print(responseVid)
        # Access the content attribute directly
        content = responseVid.content

        # Save the content to disk (e.g., as MP4)
        with open("videoTrailer_.mp4", "wb") as f:
          f.write(content)
        print("Video downloaded successfully.")
    except Exception as e:
        print(f"Error downloading video: {e}")

In [ ]:
from openai import OpenAI
import pandas as pd

# --- CONFIG --------------------------------------------------------------
OPENAI_API_KEY = userdata.get('OPENAI_API_key')
if OPENAI_API_KEY is None or OPENAI_API_KEY.startswith("INSERISCI"):
    print(f"OPENAI_API_KEY non impostata")
    raise

client = OpenAI(api_key=OPENAI_API_KEY)
page = client.videos.list()

# Analyze the response and create a table
video_data = []
for video in page.data :
    video_info = {
        'ID Video': video.id,
        'Stato': video.status,
        'Errore': video.error.message if video.error else None,
        'Scarica': ' '

    }
    video_data.append(video_info)

df_videos = pd.DataFrame(video_data)

# Display the table
display(df_videos)

# --- Download completed videos if marked ---
videos_to_download = df_videos[df_videos['Scarica'] == 'Si']

if not videos_to_download.empty:
    print("\nScaricamento dei video completati...")
    for index, row in videos_to_download.iterrows():
        video_id = row['ID Video']
        try:
            responseVid = client.videos.download_content(
                video_id=video_id,
            )
            content = responseVid.content

            # Save the content to disk (e.g., as MP4)
            with open(f"video_{video_id}.mp4", "wb") as f:
                f.write(content)
            print(f"Video '{video_id}' scaricato correttamente.")
        except Exception as e:
            print(f"Errore durante lo scaricamento del video '{video_id}': {e}")

,ID Video,Stato,Errore,Scarica
0,video_68e90cd7cd34819382f24a4918e6cd1201393bf4...,in_progress,None,
1,video_68e90b00d100819889ebf0c84f4755310508bacb...,completed,None,
2,video_68e909fd4b2c819392c8876b6a854ebd00107ca9...,completed,None,
3,video_68e9077ba858819189c292d76ef6da2609ce49e5...,completed,None,
4,video_68e8f610eaf08191a5c6abd5e7546f9807bd0b58...,completed,None,
5,video_68e8f0f68e908191b208c00071e899680df81fbb...,completed,None,
6,video_68e8ef1001a881919c8133dcd9baf5a00bcab10b...,completed,None,
7,video_68e8ea2c66448191b3a9351a59827a5a0526149e...,failed,Video generation failed due to an internal error.,
8,video_68e8e7d9b1b481919cc25a75cde50a540662b348...,completed,None,


In [ ]:
# @title SALVA SU GOOGLE DRIVE
from googleapiclient.discovery import build
from google.colab import auth
import google.auth

from googleapiclient.http import MediaFileUpload
import os

# Authenticate with Google Drive
auth.authenticate_user()
creds, _ = google.auth.default()
drive_service = build('drive', 'v3', credentials=creds)

# Specify the folder ID
folder_id = '1GKj6VKwzmY68_ZE1hHAI6iPYQTBO-fhx'

# List items within the folder, filtering by mimeType for folders
results = drive_service.files().list(
    q=f"'{folder_id}' in parents and mimeType='application/vnd.google-apps.folder' and trashed=false",
    fields="files(id, name)"
).execute()

folders = results.get('files', [])

if folders:
    print("Folders within the specified ID:")
    for idx, folder in enumerate(folders, start=1):
      print(f"{idx} - {folder['name']} (ID: {folder['id']})")

else:
    print("No folders found within the specified ID.")

print("")

while True: # Loop per richiedere input finché non è valido
        try:
            # Corrected the length check to use all_files instead of files
            target_folder = int(input(f"Inserisci il numero della folder corrente (1-{len(folders)}): "))
            # Check if the chosen numbers are within the valid range
            # Corrected the length check to use all_files instead of files
            if 1 <= target_folder <= len(folders):
                break # Valid input, exit the loop
            else:
                print(f"Errore: Inserisci numeri tra 1 e {len(folders)}.")
        except ValueError:
            print("Errore: Input non valido. Inserisci un numero intero.")

print("")

target_folder_selected = folders[target_folder - 1]
target_folder_id = target_folder_selected['id']
print(f"Folder selezionata: {target_folder_selected['name']} (ID: {target_folder_id})")

print("")
# Get the list of files and directories in the current working directory
files_and_directories = os.listdir('.')

# Filter out directories
local_files = [f for f in files_and_directories if os.path.isfile(f)]

# Print the list
print("Files in the current runtime environment:")
for item in local_files:
    print(item)


# Get the list of files and directories in the current working directory
files_and_directories = os.listdir('.')

# Filter out directories
local_files = [f for f in files_and_directories if os.path.isfile(f)]

# Iterate through each file in the list and upload it
for file_path in local_files:
    # Define the file metadata
    file_name = os.path.basename(file_path)
    file_metadata = {
        'name': file_name,
        'parents': [target_folder_id]
    }

    # Create a MediaFileUpload object
    media = MediaFileUpload(file_path, resumable=True)

    # Upload the file to the specified Google Drive folder
    try:
        uploaded_file = drive_service.files().create(
            body=file_metadata,
            media_body=media,
            fields='id'
        ).execute()

        print(f"File '{file_name}' uploaded successfully to folder with ID '{target_folder_id}'. Uploaded file ID: {uploaded_file.get('id')}")
    except Exception as e:
        print(f"Error uploading file '{file_name}': {e}")

## ☑  Fine del Processo
Tutti i file generati (testi, audio, immagini e video) sono stati sincronizzati nella cartella di destinazione su Google Drive.

**Autore e copyright** Progetto automatizzato con AI per il podcast Eugenio Barolo